# 分布训练

## 问题描述

一个7B参数，采用FB16精度的模型，仅权重就需要14GB存储。Adam优化器还要为每个参数额外存储两份，分别是一阶和二阶动量，这就有需要28GB。梯度反向传播的时候又需要14GB。

此外，前向传播中计算出来的中间值必须被保留下来，后续用于反向传播。对于词元长度2048，嵌入维度4096，单样本单层激活就需要64MB内存，堆叠32层之后，来到了2GB。批大小为8，就需要16GB了。

现在针对大尺度训练有四种策略：数据并行、张量并行、流水线并行、以及全共享数据并行。

## 基本概念

### 数据并行

最简单的并行策略。将整个模型拷贝到N张GPU，把每个训练批次切分层N等份，每个GPU跑一次前向和反向传播。反向传播执行完后，对所有GPU计算出来的梯度取平均，然后每张GPU都根据算出来的梯度更新权重，来保持所有副本权重一致。

**优点**： 吞吐线形扩展。N张GPU每步处理N倍的数据。通信只限于梯度求平均，而它能和计算重叠。

**缺点**： 每张GPU的模型都是完整的。只减轻了训练时间，每张GPU的内存没有下降。

### 张量并行

在GPU之间对单一层进行分隔。将单词的矩阵运算拆分成各个GPU上的分块运算。

**优点**：降低了每张GPU存储权重所需的内存。

**缺点**：每层之后都需要GPU之间高速通信，每次矩阵乘法之后求和增加延迟。

### 流水线并行

按层切分模型。GPU1 跑层1～8， GPU2 跑层9～16 ...。数据在流水线之间传输，GPU1 算完自己的层，将激活状态发送给GPU2，...。

**优点**：GPU之间的通行最少，只有层与层之间需要传递激活信息，相对于权重和梯度要小得多。

**缺点**：流水线旗袍。当GPU4在对批次1做前向时，GPU1、GPU2、GPU3闲着。反向传播的时候又相反。
```
           Forward             Backpropagation
GPU1   ON   (OFF   OFF       OFF    OFF)   ON
GPU2   OFF   ON    OFF       OFF    ON     OFF
GPU3   OFF   OFF   ON        ON     OFF    OFF
```